![image](image.png)

# FHIRPath Library — How It Works

Three stages to evaluate an expression against a FHIR resource:

```
string  ──►  Lexer  ──►  Token[]  ──►  Parser  ──►  AST  ──►  Interpreter  ──►  json[]
```

---
## 1. Lexer (`scanner.bal`)

Turns a raw string into a flat list of tokens.

### Token types (`token_type.bal`)

| Category | Examples |
|---|---|
| Punctuation | `(` `)` `[` `]` `.` `,` |
| Operators | `+` `-` `*` `/` `&` `\|` `~` `%` |
| Comparison | `=` `!=` `!~` `<` `>` `<=` `>=` |
| Literals | `IDENTIFIER`  `STRING`  `NUMBER`  `DATE`  `DATETIME`  `TIME` |
| Keywords | `and` `or` `xor` `implies` `true` `false` `is` `as` `in` `contains` `div` `mod` |
| Special vars | `$this`  `$index`  `$total` |

### Token structure

```ballerina
type FhirPathToken record {|
    TokenType tokenType; // category
    string    lexeme;    // raw source text
    anydata?  literal;   // interpreted value (only for literals)
    int       position;  // index in the source string
|};
```

### Example

Input: `Patient.name.where(use = 'official').given`

```
IDENTIFIER   "Patient"
DOT          "."
IDENTIFIER   "name"
DOT          "."
IDENTIFIER   "where"
LEFT_PAREN   "("
IDENTIFIER   "use"
EQUAL        "="
STRING       "'official'"  literal="official"
RIGHT_PAREN  ")"
DOT          "."
IDENTIFIER   "given"
EOF
```

---
## 2. Parser (`parser.bal` + `expr.bal`)

Turns the token list into an **Abstract Syntax Tree (AST)**.

### AST node types (`expr.bal`)

```ballerina
type Expr
    LiteralExpr          // 42  'hello'  true
  | IdentifierExpr       // name  resourceType  $this
  | MemberAccessExpr     // Patient.name
  | IndexerExpr          // name[0]
  | FunctionExpr         // where(...)  count()  first()
  | BinaryExpr           // a = b   x and y   p | q
  | UnaryExpr            // -42
  | ExternalConstantExpr // %myParam
  | QuantityLiteralExpr; // 5 'kg'
```

### AST for the example

Input: `Patient.name.where(use = 'official').given`

```
MemberAccess  .given
└── Function  where()
    ├── target:  MemberAccess  .name
    │            └── Identifier  Patient
    └── param:   Binary  =
                 ├── Identifier  use
                 └── Literal     "official"
```

---
## 3. Interpreter (`interpreter.bal`)

Walks the AST against a FHIR JSON resource and returns `json[]`.
Every result is a collection — even a single value comes back as `[value]`.

### How each node is evaluated

| Node | What it does |
|---|---|
| `Literal` | Returns `[value]` |
| `Identifier` | Looks up the field on the current context object |
| `MemberAccess` | Evaluates target, then extracts the named field from each result |
| `Indexer` | Evaluates target, picks element at the given index |
| `Function` | Evaluates target, applies the built-in function (e.g. `where`, `first`, `count`) |
| `Binary` | Evaluates both sides, applies the operator (`=`, `and`, `+`, …) |
| `Unary` | Evaluates operand, negates it |
| `ExternalConstant` | Looks up `%name` from the caller-supplied variables map |

### Walkthrough for the example

```json
{
  "resourceType": "Patient",
  "name": [
    { "use": "nickname", "given": ["Johnny"] },
    { "use": "official", "given": ["John", "Michael"] }
  ]
}
```

```
Identifier(Patient)              → [<whole resource>]
MemberAccess(.name)              → [{use:nickname,…}, {use:official,…}]
Function(where, use = 'official')→ [{use:official, given:["John","Michael"]}]
MemberAccess(.given)             → ["John", "Michael"]
```

---
## 4. Public API (`fhir_path_processor.bal`)

```ballerina
// Read
json[]|FHIRPathError getValuesFromFhirPath(json resource, string expression);

// Write
json|FHIRPathError setValuesToFhirPath(json resource, string expression, json|ModificationFunction value);
```

```ballerina
// Examples
getValuesFromFhirPath(patient, "Patient.name.where(use = 'official').given");
// → ["John", "Michael"]

getValuesFromFhirPath(patient, "Patient.name.count()");
// → [2]

setValuesToFhirPath(patient, "Patient.birthDate", "1991-01-01");
// → updated resource
```